# LangGraph와 AgentCore Memory Checkpointer(단기 메모리)

## 소개
이 Notebook에서는 **AgentCoreMemorySaver** Checkpointer를 사용하여 Amazon Bedrock AgentCore Memory 기능을 LangGraph와 통합하는 방법을 살펴봅니다. 대화 turn 전반에서 **단기 메모리**를 유지하여 자동 상태 checkpoint를 통해 Agent가 실행 중인 컨텍스트를 유지하고 이전 계산을 바탕으로 후속 계산을 수행하도록 하는 데 중점을 둡니다.

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화                                                        |
| Agent 사용 사례       | 다단계 수학 계산                                                     |
| Agentic Framework   | Langgraph                                                                        |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, Langgraph Checkpointer, Math Tools                |
| 예제 난이도  | 초급                                                                         |

다음 내용을 학습합니다.
- 자동 상태 유지를 위한 AgentCore Memory Checkpointer 생성
- AgentCore Memory backend와 LangGraph의 기본 checkpoint 시스템 사용
- 여러 상호작용에서 대화 컨텍스트 유지
- 대화 상태와 기록 확인 및 관리

### 시나리오 배경

이 예제에서는 여러 단계의 수학 계산을 수행할 수 있는 "**Math Agent**"를 만듭니다. 단순한 일회성 상호작용과 달리, 이 Agent는 AgentCore Memory의 checkpoint 기능으로 실행 중인 컨텍스트를 유지하므로 이전 계산을 바탕으로 후속 계산을 수행하고 여러 turn에 걸친 대화 흐름을 기억할 수 있습니다.

## 아키텍처
<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## 사전 요구 사항

- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory에 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델에 대한 액세스

### 통합 작동 방식

LangGraph와 AgentCore Memory의 통합은 다음과 같이 작동합니다.

1. LangGraph 상태 유지를 위한 Checkpointer backend로 AgentCore Memory 사용
2. 각 단계에서 대화 상태 자동 저장 및 불러오기
3. 여러 동시 세션과 actor 지원

이 접근 방식은 수동 메모리 작업 없이 자연스러운 상태 관리를 제공하여 유지 관리와 확장이 쉬운 Agent 아키텍처를 구현합니다.

In [ ]:
# 필요한 라이브러리 설치
!pip install -qr requirements.txt

In [ ]:
# LangGraph 및 LangChain 구성 요소 가져오기
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

In [ ]:
# Checkpointer로 사용할 AgentCoreMemorySaver 가져오기
import os
import logging

from langgraph_checkpoint_aws import AgentCoreMemorySaver
from bedrock_agentcore.memory import MemoryClient

region = os.getenv("AWS_REGION", "us-west-2")
logging.getLogger("math-agent").setLevel(logging.DEBUG)

# Memory 리소스 생성 또는 가져오기
memory_name = "MathLanggraphAgent"
client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory["id"]  # 나중에 사용할 수 있도록 이 Memory ID 유지

### AgentCore Memory 구성

이제 AgentCore Memory Checkpointer를 구성하고 LLM을 초기화합니다.

- `memory_id`는 checkpoint가 저장될 AgentCore Memory 리소스에 해당합니다.
- `region`은 리소스의 AWS 리전을 지정합니다.
- `MODEL_ID`는 LangGraph Agent를 구동할 Bedrock 모델을 정의합니다.

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# 상태 유지를 위한 Checkpointer 초기화
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

# LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

### 수학 도구

Agent가 사용할 수학 도구를 정의합니다. 이 데모에서는 두 가지 간단한 연산을 제공합니다.

In [ ]:
@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b


@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply]

### LangGraph Agent 구현

이제 AgentCore Memory Checkpointer와 LangGraph의 `create_react_agent` builder를 사용하여 Agent를 생성합니다.

In [ ]:
graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## 4단계: LangGraph Agent 실행
이제 AgentCore Memory Checkpointer가 통합된 Agent를 실행할 수 있습니다.

### 구성 설정
LangGraph에서 config는 사용자 ID나 세션 ID처럼 호출 시 필요한 속성을 담는 `RuntimeConfig`입니다. 자세한 내용은 [https://langchain-ai.github.io/langgraphjs/how-tos/configuration/](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)에서 확인할 수 있습니다.

AgentCore Memory Checkpointer(`AgentCoreMemorySaver`)에는 다음 항목을 반드시 지정해야 합니다.
- `thread_id`: AgentCore session_id(고유 대화 thread)에 매핑
- `actor_id`: AgentCore actor_id(사용자, Agent 또는 기타 식별자)에 매핑

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-1",  # 필수: 내부적으로 Bedrock AgentCore session_id에 매핑
        "actor_id": "react-agent-1",  # 필수: 내부적으로 Bedrock AgentCore actor_id에 매핑
    }
}

inputs = {
    "messages": [
        {
            "role": "user",
            "content": "What is 1337 times 515321? Then add 412 and return the value to me.",
        }
    ]
}

#### 축하합니다! Agent가 준비되었습니다!

### Agent 테스트

첫 번째 계산을 실행하여 Agent의 동작을 확인해 보겠습니다.

In [ ]:
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### Agent 상태 살펴보기

AgentCore Memory에 저장된 현재 대화 상태를 살펴보겠습니다. Checkpointer는 actor와 session의 상태를 자동으로 저장하고 검색합니다.

In [ ]:
for message in graph.get_state(config).values.get("messages"):
    print(f"{message.type}: {message.text()}")
    print("=========================================")

### Checkpoint 기록 확인

실행 중 Agent 상태가 어떻게 변했는지 checkpoint 기록을 살펴보겠습니다. Checkpoint는 역시간순으로 나열되어 가장 최근 항목이 먼저 표시됩니다.

In [ ]:
for checkpoint in graph.get_state_history(config):
    print(
        f"(Checkpoint ID: {checkpoint.config['configurable']['checkpoint_id']}) # of messages in state: {len(checkpoint.values.get('messages'))}"
    )

### 메모리 지속성 테스트

대화를 이어 가면서 Checkpointer의 기능을 테스트해 보겠습니다. Agent는 이전 계산을 기억해야 합니다.

In [ ]:
inputs = {
    "messages": [
        {
            "role": "user",
            "content": "What were the first calculations I asked you to do?",
        }
    ]
}

for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### 새 세션 시작

새 대화 thread를 만들어 세션 격리를 확인해 보겠습니다. 새 세션에서는 Agent가 이전 계산을 기억하지 못합니다.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # 새 세션 ID
        "actor_id": "react-agent-1",  # 동일한 Actor ID
    }
}

inputs = {"messages": [{"role": "user", "content": "What values did I ask you to multiply and add?"}]}
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

1. Checkpoint를 위한 AgentCore Memory 리소스를 생성하는 방법
2. 상태를 자동으로 유지하는 LangGraph Agent 구축
3. 다단계 계산을 위한 수학 도구 구현
4. AgentCoreMemorySaver를 Checkpointer backend로 사용
5. 메모리 지속성과 세션 격리 테스트

이 통합은 LangGraph의 구조화된 workflow와 AgentCore Memory의 강력한 checkpoint 기능을 결합하여 여러 상호작용에서 컨텍스트를 유지할 수 있는 상태 기반의 지속형 AI Agent를 만드는 방법을 보여 줍니다.

이 접근 방식은 Multi-Agent System, 장기 실행 workflow, 대화 컨텍스트 기반의 전문 상태 관리 등 더 복잡한 사용 사례로 확장할 수 있습니다.

### 리소스 정리
이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)